# Model selection: GGUF export of the LFM3-1.7B LoRA runs

Run 4 of the LoRA notebook trained both seeds (the merged models are in its output) but hit Kaggle's 12-hour limit during export: the importance-matrix calibration ran on the CPU (~3 h per seed). This notebook only exports: the same export_gguf.sh (f16, importance matrix on the realistic val prompts with 200 chunks, Q4_K_M, Q8_0), with llama.cpp built with CUDA so the calibration runs on the GPU. Input: the output of codeply-real-lfm3-lora.

In [ ]:
# Setup: the experiment branch (public), dependencies, the frozen split
import os, subprocess, hashlib, json
if not os.path.exists("/tmp/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "codeply-li-dtw-experiment",
                    "https://github.com/AwaisSDev/Codeply-Craft", "/tmp/codeply"], check=True)
os.chdir("/tmp/codeply/research/dtw")  # outside /kaggle/working: the output keeps only artifacts
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece pyyaml", shell=True, check=True)
subprocess.run("cd data/real && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
split = json.load(open("data/real/SPLIT.json"))
assert split["leakChecksPass"], "leak checks failed: do not train"
for k in ("train", "val", "test"):
    assert hashlib.sha256(open(f"data/real/{k}.jsonl", "rb").read()).hexdigest() == split["hashes"][k], f"{k} changed"
print({k: v["n"] for k, v in split["splits"].items()})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, glob
IMATRIX_ARGS = "-ngl 99"
if not os.path.exists("/tmp/llama.cpp/build/bin/llama-imatrix"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp && cd /tmp/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    # Export v2 failed here: ggml-cuda links CUDA::cuda_driver, which Kaggle's image does not expose to CMake.
    # That link exists only for CUDA virtual memory management (GGML_CUDA_NO_VMM=ON drops it); the driver stub
    # directory is given as well. If the CUDA build still fails, fall back to the CPU build (~3 h per seed).
    has_gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
    print("GPU:", has_gpu)
    # No GPU (e.g. a CPU session once the weekly GPU quota is used up): skip the CUDA build entirely.
    cuda = subprocess.run("false" if not has_gpu else "cd /tmp/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF "
                          "-DLLAMA_BUILD_EXAMPLES=OFF -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
                          "-DCMAKE_LIBRARY_PATH=/usr/local/cuda/lib64/stubs && "
                          "cmake --build build -j4 --target llama-quantize llama-imatrix", shell=True, capture_output=True, text=True)
    if cuda.returncode:
        print("CUDA build failed, building for CPU:", cuda.stdout[-1500:], cuda.stderr[-1500:])
        IMATRIX_ARGS = ""
        subprocess.run("cd /tmp/llama.cpp && rm -rf build && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF "
                       "-DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize llama-imatrix > /dev/null",
                       shell=True, check=True)
print("imatrix args:", repr(IMATRIX_ARGS))
subprocess.run("pip -q install -e /tmp/llama.cpp/gguf-py", shell=True, check=True)
subprocess.run("node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs) > /dev/null", shell=True, check=True)
os.chdir("/tmp/codeply")
# Calibration text = the realistic val prompts, as in run 4 (export_gguf.sh reads ml/data/out/sft-val.jsonl).
subprocess.run("cp research/dtw/data/real/train.jsonl research/dtw/data/real/val.jsonl /tmp/ && node ml/data/export-sft.mjs --in /tmp --style full "
               "&& cp /tmp/sft-val.jsonl ml/data/out/sft-val.jsonl", shell=True, check=True)
MERGE = '''
import glob, subprocess, sys, torch, yaml
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
sys.path.insert(0, "/tmp/codeply/ml/finetune")
from train_lora import copy_base_tokenizer
base = yaml.safe_load(open("/tmp/codeply/ml/finetune/config-lfm3.yaml"))["base_model"]
for ad in sys.argv[1:]:
    name = [x for x in ad.split("/") if x.startswith("real-lfm3-lora-s")][-1]
    m = f"/tmp/merged/{name}/merged"
    # LFM3 revision current when run 4 trained (latest commit 2026-10-28; training started 2026-11-01).
    model = AutoModelForCausalLM.from_pretrained(base, revision="ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7", dtype=torch.float16, trust_remote_code=True)
    model = PeftModel.from_pretrained(model, ad).merge_and_unload()
    model.save_pretrained(m, safe_serialization=True)
    AutoTokenizer.from_pretrained(base, revision="ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7", trust_remote_code=True).save_pretrained(m)
    copy_base_tokenizer(base, m)
    subprocess.run(f"cp {ad}/../train-log.jsonl /tmp/merged/{name}/ 2>/dev/null", shell=True)
    print("merged", name, flush=True)
    del model
'''
merged = sorted(glob.glob("/kaggle/input/**/real-lfm3-lora-s*/merged", recursive=True))
if not merged:
    # Kaggle does not accept a cancelled run as a kernel source, so the adapters come in as a private dataset
    # (codeply-real-lfm3-lora-adapters: the final adapter of each seed from run 4). Merge them exactly as
    # ml/finetune/train_lora.py does: base in fp16 on the T4, PEFT merge_and_unload, base tokenizer files.
    # The merge runs in a fresh Python process: export v3 failed importing peft/transformers in the notebook's
    # own process after the pip installs above had changed numpy under it.
    # The dataset may hold one archive per seed (kaggle datasets create -r tar); unpack them if so.
    os.makedirs("/tmp/inp", exist_ok=True)
    for a in glob.glob("/kaggle/input/**/*.tar", recursive=True) + glob.glob("/kaggle/input/**/*.zip", recursive=True):
        subprocess.run(f"cd /tmp/inp && (tar -xf {a} 2>/dev/null || unzip -oq {a})", shell=True)
    adapters = sorted(set(glob.glob("/kaggle/input/**/real-lfm3-lora-s*/adapter", recursive=True) + glob.glob("/tmp/inp/**/adapter", recursive=True)))
    print("adapters:", adapters)
    assert adapters, "no adapters in the input"
    # As in the LoRA notebook (run 1): Kaggle ships torchao 0.10, which PEFT rejects when it loads LoRA layers.
    # The same library versions as the training run (LoRA notebook): export v6 got the newest transformers,
    # whose built-in LFM class rejects LFM3's layer types; transformers 5.22.1 loads LFM3's own code.
    subprocess.run("cd /tmp/codeply && pip -q install -r ml/finetune/requirements.txt", shell=True, check=True)
    subprocess.run("pip -q uninstall -y torchao", shell=True)
    open("/tmp/merge_lora.py", "w").write(MERGE)
    if subprocess.run("python -c 'import numpy, torch, transformers, peft'", shell=True).returncode:
        subprocess.run("pip -q install --force-reinstall --no-deps numpy==$(python -c 'import importlib.metadata as m; print(m.version(\"numpy\"))')", shell=True)
    subprocess.run("python /tmp/merge_lora.py " + " ".join(adapters), shell=True, check=True)
    merged = sorted(glob.glob("/tmp/merged/real-lfm3-lora-s*/merged"))
print(merged)
assert merged, "no merged models and no adapters in the input"
for m in merged:
    name = m.split("/")[-2]
    out = f"/kaggle/working/runs/{name}"
    os.makedirs(out, exist_ok=True)
    if glob.glob(f"{out}/*Q4_K_M.gguf"):
        continue
    subprocess.run(f"cd /tmp/codeply && OUT=/tmp/gguf-{name} IMATRIX_ARGS='{IMATRIX_ARGS}' LLAMA_CPP=/tmp/llama.cpp "
                   f"bash ml/finetune/export_gguf.sh {m} {name}", shell=True, check=True)
    for f in glob.glob(f"/tmp/gguf-{name}/*Q4_K_M.gguf") + glob.glob(f"/tmp/gguf-{name}/*Q8_0.gguf") + glob.glob(f"{m}/../train-log.jsonl"):
        subprocess.run(f"cp {f} {out}/", shell=True, check=True)
    subprocess.run(f"rm -rf /tmp/gguf-{name}", shell=True, check=True)

In [ ]:
!ls -laR /kaggle/working/runs